# Tutorial 3 — Variational Problems, Spectra, and Linear Models on Geometric Data

**MM845 — Tópicos de Geometria III: AI for Geometry**
Paired with **Lecture 3: Linear Models, Optimisation & Regularisation**

---

Lecture 3 is the one lecture where everything can be proved: least squares is an
orthogonal projection, the loss is convex, the minimiser is unique and available in
closed form, and the convergence rate of gradient descent is a ratio of eigenvalues.
This tutorial spends that certainty on geometry.

It has two halves, and they are the same idea twice.

**Part I — approximation on a manifold as energy minimisation.** Fitting a function
on $S^2$ *is* orthogonal projection in $L^2(S^2)$. Choose the eigenbasis of the
Laplace–Beltrami operator and the projection becomes diagonal, the condition number
drops to $1$, and Tikhonov regularisation becomes literally a penalty on the
Dirichlet energy $\int_M|\nabla f|^2$.

**Part II — linear models on a dataset of geometric objects.** We build random plane
curves, label them with quantities we can compute exactly, and predict those labels
by linear and logistic regression. The theme is Lecture 3's design principle: *the
feature map is worth more than the model*.

| § | Topic | Lecture 3 |
|---|---|---|
| **I** | | |
| 1 | Least squares as orthogonal projection; empirical vs exact inner product | slide 3 |
| 2 | The Laplace–Beltrami eigenbasis, and why the basis decides $\kappa$ | slides 5–6, 9 |
| 3 | Recovering that eigenbasis from a bare point cloud | — |
| 4 | Spectral decay, and ridge $=$ Dirichlet energy | slides 3, 8 |
| **II** | | |
| 5 | Random plane curves; symmetry and invariant features | slides 9, 10 |
| 6 | Regression: area (exactly) and length (identifiably) | slides 3, 5–6, 11 |
| 7 | Classification, ridge and lasso | slides 4, 8 |
| 8 | PyTorch: the canonical training loop | slide 11 |

You need the `aigeo` environment from [Tutorial 1](../tutorial_01/README.md);
PyTorch is used in §8 only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from itertools import combinations_with_replacement

SEED = 20260817
rng = np.random.default_rng(SEED)

GEO_DARK, GEO_TEAL, GEO_RUST = "#103158", "#006c86", "#b2461e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[GEO_DARK, GEO_TEAL, GEO_RUST,
                                         "#6a8caf", "#4c9a2a", "#7d3c98"]),
})
print("numpy", np.__version__)

In [ ]:
# ---- carried over from Tutorial 2 -------------------------------------------
GOLDEN = (1 + 5**0.5) / 2


def fibonacci_sphere(n_points):
    """Deterministic quasi-uniform point set on S^2 (a good quadrature rule)."""
    i = np.arange(n_points)
    z = 1 - 2 * (i + 0.5) / n_points
    r = np.sqrt(np.maximum(0.0, 1 - z**2))
    theta = 2 * np.pi * i / GOLDEN
    return np.stack([r * np.cos(theta), r * np.sin(theta), z], axis=1)


def sample_sphere(n_points, dim, rng):
    """i.i.d. uniform samples on S^{dim-1}."""
    x = rng.normal(size=(n_points, dim))
    return x / np.linalg.norm(x, axis=1, keepdims=True)


def sphere_ax(fig, pos, title="", elev=22, azim=35, lim=1.05):
    ax = fig.add_subplot(pos, projection="3d")
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
    ax.view_init(elev=elev, azim=azim)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_title(title); ax.grid(False)
    return ax


def poly_features(X, degree):
    """Monomials of degree <= `degree`, with their degrees."""
    cols, deg = [np.ones(len(X))], [0]
    for d in range(1, degree + 1):
        for combo in combinations_with_replacement(range(X.shape[1]), d):
            cols.append(np.prod(X[:, combo], axis=1)); deg.append(d)
    return np.stack(cols, axis=1), np.array(deg)

---
# Part I — Approximation on a manifold as energy minimisation

---
## 1. Least squares is an orthogonal projection

Let $M$ be a compact Riemannian manifold with normalised volume measure $\mu$, and
let $V = \operatorname{span}(\varphi_1,\dots,\varphi_m) \subset L^2(M,\mu)$ be a
finite-dimensional *ansatz space*. Given a target $g$, minimise the energy

$$E(f) \;=\; \int_M \big(f(x) - g(x)\big)^2 \, d\mu(x), \qquad f \in V .$$

Writing $f = \sum_j \theta_j \varphi_j$ and differentiating gives the **normal
equations** in exactly Lecture 3's form,

$$G\theta = c, \qquad G_{jk} = \langle \varphi_j, \varphi_k\rangle_{L^2}, \quad c_j = \langle \varphi_j, g\rangle_{L^2},$$

with $G$ the **Gram matrix**. The minimiser is the orthogonal projection $P_V g$,
and the residual is orthogonal to $V$ — invisible to the model, by construction.

**What changes when you only have samples.** We do not have $\mu$; we have points.
Replacing the integral by an average over $N$ samples turns the $L^2$ inner product
into the **empirical** one, $\langle u, v \rangle_N = \frac1N \sum_i u(x_i)v(x_i)$,
and the problem becomes ordinary least squares with design matrix
$X_{ij} = \varphi_j(x_i)$.

So **the design matrix is a discretised Gram matrix**, $\frac1N X^\top X \to G$, and
Lecture 2's "empirical risk approximates true risk" is here the statement that one
inner product approximates another, at the Monte-Carlo rate $O(N^{-1/2})$.

In [ ]:
def empirical_gram(F):
    """Gram matrix under the empirical inner product <f,h>_N = mean(f*h)."""
    return (F.T @ F) / len(F)


# three simple functions on S^2, orthogonal in L^2 (they are harmonics)
def trio(X):
    x, y, z = X.T
    return np.stack([np.ones_like(x), z, x * y], axis=1)


G_exact = np.diag([1.0, 1 / 3, 1 / 15])     # <1,1>=1, <z,z>=1/3, <xy,xy>=1/15
print("exact Gram (diagonal):", np.diag(G_exact))

# a single draw is noisy, so average the error over 25 independent samples
for N in [10**2, 10**3, 10**4, 10**5]:
    errs = [np.abs(empirical_gram(trio(sample_sphere(N, 3, rng))) - G_exact).max()
            for _ in range(25)]
    print(f"  N = {N:6d}   mean max|G_N - G| = {np.mean(errs):.2e}"
          f"    (1/sqrt(N) = {N**-0.5:.2e})")

print("\ni.i.d. sampling vs Fibonacci quadrature, same number of points:")
for N in [10**3, 10**4]:
    e_mc = np.abs(empirical_gram(trio(sample_sphere(N, 3, rng))) - G_exact).max()
    e_fb = np.abs(empirical_gram(trio(fibonacci_sphere(N))) - G_exact).max()
    print(f"  N = {N:6d}   i.i.d. {e_mc:.2e}     Fibonacci {e_fb:.2e}")

The error tracks $N^{-1/2}$, as Monte Carlo must — and the Fibonacci lattice of
Tutorial 2 §2 is *far* better than i.i.d. sampling for smooth integrands. Two things
to carry forward: a basis orthonormal in $L^2$ is only approximately orthonormal on
your sample, and quadrature beats sampling whenever you are allowed to choose the
points.

> **Exercise 1 — the projection theorem, numerically.**
> (a) With $V$ spanned by the three functions above and target $g(x,y,z)=z^3$,
> compute $P_V g$ from the normal equations and verify the residual is orthogonal to
> each $\varphi_j$ to within Monte-Carlo error.
>
> (b) Verify $\lVert g\rVert^2 = \lVert P_V g\rVert^2 + \lVert g - P_V g\rVert^2$.
> Which of Lecture 2's error terms does each piece correspond to?

---
## 2. The eigenbasis of the Laplace–Beltrami operator

Any basis of $V$ gives the same subspace and the same projection. But not the same
*numerics*, and one basis is canonically better than the rest.

On $S^2$ the operator $\Delta_{S^2}$ has eigenspaces $\mathcal{H}_\ell$ — the
**spherical harmonics** — with

$$\Delta_{S^2} Y = -\ell(\ell+1)\, Y, \qquad \dim \mathcal{H}_\ell = 2\ell+1,
\qquad L^2(S^2) = \bigoplus_{\ell \ge 0} \mathcal{H}_\ell .$$

Rather than write them out, we **construct** them. Restricting monomials of degree
$\le L$ to the sphere and running Gram–Schmidt *in order of degree* produces an
orthonormal basis automatically stratified by $\ell$: whatever survives
orthogonalisation at each new degree is exactly $\mathcal{H}_\ell$. Monomials killed
by the relation $x^2+y^2+z^2=1$ have zero residual and are dropped.

In [ ]:
def harmonic_transform(L, Xq, tol=1e-8):
    """Orthonormal basis of polynomials of degree <= L on S^2, stratified by degree.

    Returns (T, ell): the basis functions are `poly_features(X, L)[0] @ T`, and
    ell[j] is the harmonic degree of basis function j.
    """
    Phi, mono_deg = poly_features(Xq, L)
    U = Phi / np.sqrt(len(Xq))          # so Euclidean orthonormality on U == L^2
    Q, T, ell = [], [], []
    for j in range(U.shape[1]):
        v = U[:, j].copy()
        t = np.zeros(U.shape[1]); t[j] = 1.0
        for q, tq in zip(Q, T):         # modified Gram-Schmidt
            c = q @ v
            v -= c * q
            t -= c * tq
        nrm = np.linalg.norm(v)
        if nrm < tol:                   # dependent: killed by x^2+y^2+z^2-1
            continue
        Q.append(v / nrm); T.append(t / nrm); ell.append(mono_deg[j])
    return np.array(T).T, np.array(ell)


L_MAX = 6
Xq = fibonacci_sphere(20000)            # quadrature set
T, ELL = harmonic_transform(L_MAX, Xq)


def harmonics(X, T=T, L=L_MAX):
    """Evaluate the orthonormal harmonic basis at arbitrary points."""
    return poly_features(X, L)[0] @ T


print("basis functions kept, by degree:")
for d in range(L_MAX + 1):
    print(f"   l = {d}:  {int((ELL == d).sum()):2d} functions   (2l+1 = {2*d+1})")
print(f"total {len(ELL)}   =  (L+1)^2 = {(L_MAX+1)**2}")

Y = harmonics(Xq)
G = empirical_gram(Y)
print(f"orthonormal on the quadrature set : max|G - I| = "
      f"{np.abs(G - np.eye(len(ELL))).max():.2e}   (exact by construction)")

Y_mc = harmonics(sample_sphere(200000, 3, rng))
G_mc = empirical_gram(Y_mc)
print(f"on a fresh i.i.d. sample          : max|G - I| = "
      f"{np.abs(G_mc - np.eye(len(ELL))).max():.2e}   (Monte-Carlo error)")

The multiplicities come out $1, 3, 5, 7, 9, 11, 13$ without being asked for. That is
$\dim\mathcal{H}_\ell = 2\ell+1$ — the first sign that the construction found the
right objects rather than merely *some* orthonormal basis. The second line is §1's
lesson again.

In [ ]:
Xv = fibonacci_sphere(8000)
Yv = harmonics(Xv)
show = [0, 1, 3, 4, 8, 9, 15, 16]        # a few from l = 0,1,2,3,4

fig = plt.figure(figsize=(11.0, 5.4))
for k, j in enumerate(show):
    ax = sphere_ax(fig, 241 + k, f"$\\ell = {ELL[j]}$", elev=20)
    v = Yv[:, j]
    m = np.abs(v).max()
    ax.scatter(*Xv.T, c=v, cmap="coolwarm", vmin=-m, vmax=m, s=4, lw=0)
fig.suptitle("Spherical harmonics, obtained by Gram–Schmidt on monomials", y=1.0)
plt.tight_layout(); plt.show()

### Why the basis decides how hard the problem is

Lecture 3 defined $\kappa = \lambda_{\max}/\lambda_{\min}$ of $X^\top X$ and made two
claims: it is the eccentricity of the elliptical level sets, and it controls gradient
descent through $(1-1/\kappa)^t$. The monomial and harmonic bases span **the same
subspace** of $L^2(S^2)$ — same projection, same minimum, different coordinates.

In [ ]:
Xc = sample_sphere(4000, 3, rng)
Phi_mono = poly_features(Xc, L_MAX)[0]
Phi_harm = harmonics(Xc)


def kappa(F):
    sv = np.linalg.svd(F, compute_uv=False)
    sv = sv[sv > sv[0] * 1e-14]           # ignore the exact rank deficiency
    return (sv[0] / sv[-1])**2


k_mono, k_harm = kappa(Phi_mono), kappa(Phi_harm)
print(f"monomials  (84 columns, rank 49):  kappa = {k_mono:.3e}")
print(f"harmonics  (49 columns, rank 49):  kappa = {k_harm:.3e}")
print(f"\ngradient descent contraction per step, (1 - 1/kappa):")
print(f"   monomials {1 - 1/k_mono:.10f}      harmonics {1 - 1/k_harm:.6f}")
print(f"steps to reduce the error by 10x:")
print(f"   monomials {np.log(10)*k_mono:,.0f}      harmonics {np.log(10)*k_harm:.1f}")

Same subspace, same answer, and orders of magnitude difference in how hard it is to
*find* that answer by gradient descent. In the harmonic basis $\kappa \approx 1$: the
level sets are round. In the monomial basis they are enormously eccentric and the
descent crawls along a valley — Lecture 3's zig-zag picture.

This is the practical content of "choose a good basis", and it recurs all course.
Normalising features (Lecture 2 §11) is the cheap version; orthogonalising against
the geometry is the expensive, better one. In Lecture 10 it becomes the statement
that a symmetry-adapted basis is a better hypothesis space.

---
## 3. Recovering the eigenbasis from the point cloud alone

So far we used the fact that we know $S^2$ is the unit sphere. Suppose we did not.

Build a weighted graph on the samples with heat-kernel weights
$W_{ij} = \exp(-\lVert x_i - x_j\rVert^2/4t)$, let $D = \operatorname{diag}(W\mathbf{1})$,
and form the **random-walk graph Laplacian** $L_t = \frac1t(I - D^{-1}W)$. As
$t \to 0$ and $N \to \infty$ this converges to $\Delta_M$ (Belkin–Niyogi). Nothing in
it refers to the sphere — only distances between samples.

In [ ]:
def graph_laplacian_spectrum(X, t, n_eig=30):
    """Eigenvalues and eigenvectors of the random-walk graph Laplacian."""
    D2 = np.maximum(0.0, 2 - 2 * np.clip(X @ X.T, -1, 1))   # squared distances
    W = np.exp(-D2 / (4 * t))
    np.fill_diagonal(W, 0.0)
    d = W.sum(axis=1)
    Dm = 1 / np.sqrt(d)
    S = (Dm[:, None] * W) * Dm[None, :]     # symmetric conjugate: same spectrum
    evals, evecs = np.linalg.eigh(S)
    evals, evecs = evals[::-1], evecs[:, ::-1]
    lam = (1 - evals) / t
    return lam[:n_eig], (Dm[:, None] * evecs[:, :n_eig])


Xg = fibonacci_sphere(2500)
lam, phi = graph_laplacian_spectrum(Xg, t=0.005)

# fix the overall scale using the first non-trivial cluster, which must be l(l+1)=2
lam_scaled = lam / (lam[1:4].mean() / 2)

print("  k   eigenvalue    nearest l(l+1)")
for k in range(17):
    print(f"  {k:2d}   {lam_scaled[k]:8.3f}      {round(np.sqrt(max(lam_scaled[k],0)+0.25)-0.5)}")

Read the left column downwards: one zero, then **three** near $2$, **five** near $6$,
**seven** near $12$, **nine** near $20$. Those are $\ell(\ell+1)$ with multiplicities
$2\ell+1$.

The multiplicities are the convincing part: they are integers, so no constant can
fudge them, and they are forced by the representation theory of $SO(3)$ acting on
$L^2(S^2)$ — reproduced by a graph built from distances alone. The eigenvalues
themselves are biased low, and the bias is $O(t)$.

In [ ]:
ts = [0.08, 0.04, 0.02, 0.01, 0.005, 0.0025]
targets = {2: 6.0, 3: 12.0, 4: 20.0}
blocks = {2: slice(4, 9), 3: slice(9, 16), 4: slice(16, 25)}
errs = {l: [] for l in targets}

for t in ts:
    lam_t, _ = graph_laplacian_spectrum(Xg, t=t)
    s = lam_t / (lam_t[1:4].mean() / 2)
    for l in targets:
        errs[l].append(abs(s[blocks[l]].mean() - targets[l]) / targets[l])

fig, ax = plt.subplots(figsize=(5.4, 3.4))
for l, col in zip(targets, [GEO_DARK, GEO_TEAL, GEO_RUST]):
    ax.loglog(ts, errs[l], "o-", color=col, label=f"$\\ell = {l}$")
ref = np.array(ts) * (errs[2][0] / ts[0])
ax.loglog(ts, ref, "k--", lw=1, alpha=0.6, label=r"slope 1, i.e. $O(t)$")
ax.set_xlabel("bandwidth $t$"); ax.set_ylabel("relative error in $\\lambda_\\ell$")
ax.set_title("the graph Laplacian converges to $\\Delta_{S^2}$ at rate $O(t)$")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

for l in targets:
    print(f"l = {l}: relative error {errs[l][0]:.1%} at t={ts[0]}  ->  "
          f"{errs[l][-1]:.1%} at t={ts[-1]}")

Halving $t$ halves the error — clean first-order convergence, with a constant growing
with $\ell$ because higher modes need a narrower kernel.

> **Exercise 2 — the graph Laplacian is doing geometry.**
> (a) Squash the sphere into an ellipsoid $x^2/a^2+y^2+z^2=1$ with $a=1.4$. The
> $SO(3)$ symmetry breaks, so the $(2\ell+1)$-fold degeneracies must split. Watch
> them split, and identify the residual symmetry protecting those that remain.
>
> (b) Estimate the area of an unknown surface from **Weyl's law**,
> $\lambda_k \sim 4\pi k/\operatorname{Area}(M)$ in dimension 2. Test on $S^2$, where
> the answer is $4\pi$.

---
## 4. Spectral decay, and ridge as a penalty on the Dirichlet energy

Because the basis is orthonormal, the truncation error to degree $L$ is exactly the
tail, $\lVert g - P_{V_L}g\rVert^2 = \sum_{\ell>L}\sum_m |c_{\ell m}|^2$. So
**approximation error is coefficient decay**, and coefficient decay is smoothness —
the manifold version of a fact every analyst knows for Fourier series.

In [ ]:
def g_smooth(X):                 # a fixed low-degree harmonic combination: analytic
    x, y, z = X.T
    return 0.6 * (2 * z**2 - x**2 - y**2) + 1.1 * x * y + 0.9 * (x**3 - 3 * x * y**2)


BUMP = np.array([0.4, 0.5, 0.76]); BUMP /= np.linalg.norm(BUMP)   # off-axis, so all m appear


def g_gaussian(X, x0=BUMP, w=0.35):
    return np.exp(-(2 - 2 * (X @ x0)) / (2 * w**2))          # smooth bump, all degrees


def g_kink(X):                   # continuous but not differentiable: |z|
    return np.abs(X[:, 2])


TARGETS = [("harmonic (degree 3)", g_smooth), ("Gaussian bump", g_gaussian),
           ("$|z|$ (a kink)", g_kink)]

# coefficients by quadrature: c_j = <g, Y_j>
Yq = harmonics(Xq)
fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.2))
for (name, g), col in zip(TARGETS, [GEO_DARK, GEO_TEAL, GEO_RUST]):
    c = (Yq.T @ g(Xq)) / len(Xq)
    power = np.array([np.sum(c[ELL == l]**2) for l in range(L_MAX + 1)])
    tail = np.array([power[l+1:].sum() for l in range(L_MAX + 1)])
    axes[0].semilogy(np.arange(L_MAX + 1), np.maximum(power, 1e-18), "o-", color=col, label=name)
    axes[1].semilogy(np.arange(L_MAX + 1), np.maximum(tail, 1e-18), "o-", color=col, label=name)
axes[0].set_xlabel("degree $\\ell$"); axes[0].set_ylabel("power in $\\mathcal{H}_\\ell$")
axes[0].set_title("energy per eigenspace"); axes[0].legend(fontsize=7)
axes[1].set_xlabel("truncation degree $L$"); axes[1].set_ylabel("squared tail")
axes[1].set_title("truncation error $\\|g - P_{V_L}g\\|^2$")
plt.tight_layout(); plt.show()

Three regimes, exactly as the theory says. The **harmonic** target is *in* $V_3$, so
its tail is zero to machine precision. The **Gaussian bump** is analytic and its
coefficients decay geometrically — spectral accuracy. The **kink** $|z|$ is only
Lipschitz and decays like a power of $\ell$: doubling the degree buys a fixed factor,
not a fixed number of digits.

### Regularisation is geometry

The **Dirichlet energy** is $E_{\mathrm{Dir}}(f) = \int_M|\nabla f|^2 = \langle f, -\Delta f\rangle
= \sum_k \lambda_k c_k^2$ with $\lambda_k = \ell_k(\ell_k+1)$. So in the harmonic
basis it is a *weighted $\ell^2$ norm of the coefficients*, and the penalised problem

$$\min_{c} \; \tfrac1N\lVert Yc - y\rVert^2 \;+\; \lambda \sum_k \lambda_k c_k^2$$

is three things at once: weighted ridge regression; Tikhonov regularisation of an
ill-posed inverse problem; and "fit the data but keep the surface taut". Smoothness
*is* the regulariser. The solution is a **spectral filter**,
$\hat c_k = \hat c_k^{\,\mathrm{LS}}/(1 + \lambda \lambda_k)$ — high frequencies
suppressed hardest.

That closed form needs the empirical Gram to be the identity, which §1 said holds
only up to sampling error. So we place the training points on a Fibonacci lattice
rather than drawing them i.i.d. — legitimate when designing an experiment, and worth
two orders of magnitude here.

In [ ]:
def fit_weighted_ridge(Yd, yd, lam, weights):
    """Minimise (1/N)||Y c - y||^2 + lam * sum_k w_k c_k^2   (closed form)."""
    N = len(Yd)
    A = (Yd.T @ Yd) / N + lam * np.diag(weights)
    return np.linalg.solve(A, (Yd.T @ yd) / N)


N_TR, NOISE = 220, 0.35
X_tr = fibonacci_sphere(N_TR)      # a quadrature rule: see the note below
X_te = fibonacci_sphere(20000)
Y_tr, Y_te = harmonics(X_tr), harmonics(X_te)

print(f"empirical Gram of the 49 harmonics on the {N_TR} training points:")
print(f"   Fibonacci  max|G - I| = "
      f"{np.abs(empirical_gram(harmonics(fibonacci_sphere(N_TR))) - np.eye(len(ELL))).max():.2e}")
print(f"   i.i.d.     max|G - I| = "
      f"{np.abs(empirical_gram(harmonics(sample_sphere(N_TR, 3, rng))) - np.eye(len(ELL))).max():.2e}")

g_true = g_gaussian
y_tr = g_true(X_tr) + NOISE * rng.normal(size=N_TR)
y_te = g_true(X_te)                       # noiseless: we measure approximation error

LAM_DIR = ELL * (ELL + 1.0)               # Laplacian eigenvalues
w_none = np.zeros(len(ELL))
w_ridge = np.ones(len(ELL))

lams = np.logspace(-6, 1, 60)
curves = {}
for name, w in [("ridge  ($w_k = 1$)", w_ridge), ("Dirichlet  ($w_k = \\lambda_k$)", LAM_DIR)]:
    curves[name] = [np.mean((Y_te @ fit_weighted_ridge(Y_tr, y_tr, l, w) - y_te)**2)
                    for l in lams]
mse_ls = np.mean((Y_te @ fit_weighted_ridge(Y_tr, y_tr, 0.0, w_none) - y_te)**2)
baseline = np.mean((y_te - y_tr.mean())**2)

fig, ax = plt.subplots(figsize=(5.8, 3.4))
for (name, cv), col in zip(curves.items(), [GEO_TEAL, GEO_DARK]):
    ax.loglog(lams, cv, "-", color=col, lw=1.8, label=name)
ax.axhline(mse_ls, color=GEO_RUST, ls="--", lw=1.3, label="unregularised")
ax.axhline(baseline, color="0.5", ls=":", lw=1.3, label="predict the mean")
ax.set_xlabel("$\\lambda$"); ax.set_ylabel("test MSE")
ax.set_title(f"$N_{{train}} = {N_TR}$, noise $\\sigma = {NOISE}$")
ax.legend(fontsize=7.5)
plt.tight_layout(); plt.show()

for name, cv in curves.items():
    print(f"{name:34s} best test MSE {min(cv):.5f} at lambda = {lams[int(np.argmin(cv))]:.2e}")
print(f"{'unregularised least squares':34s} test MSE {mse_ls:.5f}")
print(f"{'predict-the-mean baseline':34s} test MSE {baseline:.5f}")

Both penalties beat the unregularised fit, and the **Dirichlet penalty wins** —
because it uses information plain ridge does not have. Plain ridge treats all $49$
coefficients as equally suspect; the Dirichlet penalty knows the noise is spread
evenly across the spectrum while the signal is concentrated at low $\ell$. That is a
geometric prior, in Lecture 2's precise sense.

In [ ]:
lam_star = lams[int(np.argmin(curves["Dirichlet  ($w_k = \\lambda_k$)"]))]
c_ls = fit_weighted_ridge(Y_tr, y_tr, 0.0, w_none)
c_dir = fit_weighted_ridge(Y_tr, y_tr, lam_star, LAM_DIR)
c_pred = c_ls / (1 + lam_star * LAM_DIR)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.3))
big = np.abs(c_ls) > 0.05 * np.abs(c_ls).max()      # ratios are meaningless where c_ls ~ 0
axes[0].plot(LAM_DIR[big], (c_dir / c_ls)[big], "o", color=GEO_DARK, ms=5, label="measured")
xs = np.linspace(0, LAM_DIR.max(), 200)
axes[0].plot(xs, 1 / (1 + lam_star * xs), "-", color=GEO_RUST, lw=1.6,
             label=r"$1/(1+\lambda\lambda_k)$")
axes[0].set_xlabel(r"$\lambda_k = \ell(\ell+1)$"); axes[0].set_ylabel("shrinkage factor")
axes[0].set_ylim(0, 1.15)
axes[0].set_title("regularisation acts as a spectral filter"); axes[0].legend(fontsize=8)

c_true = (harmonics(Xq).T @ g_true(Xq)) / len(Xq)
axes[1].semilogy(np.arange(len(ELL)), np.abs(c_true), "o", ms=3.5, color="0.6", label="truth")
axes[1].semilogy(np.arange(len(ELL)), np.abs(c_ls), "^", ms=3.5, color=GEO_RUST, label="unregularised")
axes[1].semilogy(np.arange(len(ELL)), np.abs(c_dir), "s", ms=3.5, color=GEO_DARK, label="Dirichlet")
axes[1].set_xlabel("basis index $k$ (ordered by $\\ell$)"); axes[1].set_ylabel("$|c_k|$")
axes[1].set_title("high-frequency noise is suppressed"); axes[1].legend(fontsize=7.5)
plt.tight_layout(); plt.show()

print(f"closed form matches solver: max |c_dir - c_ls/(1+lam*lam_k)| = "
      f"{np.abs(c_dir - c_pred).max():.2e}")

The measured shrinkage factors fall exactly on $1/(1+\lambda\lambda_k)$, and the
high-frequency coefficients are visibly suppressed.

> **Exercise 3 — the regulariser encodes a belief, and the basis sets $\kappa$.**
> (a) Penalise $\int(\Delta f)^2$ instead, i.e. $w_k = \lambda_k^2$. Derive and verify
> the new filter. When would you prefer it?
>
> (b) Make the *noise* smooth — built only from harmonics of degree $\le 2$. The
> Dirichlet penalty should lose its advantage. Explain why in one sentence, and say
> what that implies about geometric priors generally.
>
> (c) Run plain gradient descent on both design matrices of §2 with $\eta = 1/\lambda_{\max}$
> and plot loss against iteration. Confirm the rate $(1-1/\kappa)^t$, then apply
> feature standardisation to the monomials and measure how much of the gap it closes.

---
# Part II — Linear models on a dataset of geometric objects

---
## 5. Random plane curves, and what a feature map is for

Each data point is now *an entire geometric object*. Take closed plane curves in
polar form,

$$r(\theta) \;=\; a_0 + \sum_{k=1}^{K}\big(a_k \cos k\theta + b_k \sin k\theta\big),$$

with $a_0 = 1$ and $a_k, b_k \sim \mathcal{N}(0, \sigma^2 k^{-2s})$ — the decay
exponent $s$ controlling smoothness exactly as in §4, now on $S^1$.

Three labels, all computable exactly: the **area**, the **length**
$\int_0^{2\pi}\sqrt{r^2+r'^2}\,d\theta$ by quadrature, and **convexity** from the
sign of $r^2 + 2r'^2 - rr''$. The area is worth pausing on — by Parseval,

$$A \;=\; \tfrac12\int_0^{2\pi} r^2 d\theta \;=\; \pi a_0^2 + \frac{\pi}{2}\sum_{k=1}^{K}\big(a_k^2 + b_k^2\big),$$

an *exactly quadratic* function of the coefficients. §6 turns that into the sharpest
demonstration in this tutorial.

In [ ]:
K_MODES, DECAY, SIGMA = 8, 2.5, 0.25
M_QUAD = 512
TH = np.linspace(0, 2 * np.pi, M_QUAD, endpoint=False)
KS = np.arange(1, K_MODES + 1)
_ANG = np.outer(KS, TH)
_COS, _SIN = np.cos(_ANG), np.sin(_ANG)


def make_curves(n, rng, sigma=SIGMA, s=DECAY):
    """Random star-shaped plane curves, with exact geometric labels."""
    scale = sigma / KS**s
    a = rng.normal(size=(n, K_MODES)) * scale
    b = rng.normal(size=(n, K_MODES)) * scale
    r = 1 + a @ _COS + b @ _SIN
    r1 = (-a * KS) @ _SIN + (b * KS) @ _COS
    r2 = (-a * KS**2) @ _COS + (-b * KS**2) @ _SIN
    area = np.pi + 0.5 * np.pi * np.sum(a**2 + b**2, axis=1)      # exact
    length = np.mean(np.sqrt(r**2 + r1**2), axis=1) * 2 * np.pi   # quadrature
    convex = ((r**2 + 2 * r1**2 - r * r2).min(axis=1) >= 0).astype(float)
    return dict(a=a, b=b, r=r, area=area, length=length, convex=convex)


D = make_curves(6000, rng)
print(f"curves: {len(D['area'])}    all r > 0: {bool((D['r'] > 0).all())}")
print(f"convex fraction {D['convex'].mean():.1%}   "
      f"-> majority-class baseline {max(D['convex'].mean(), 1-D['convex'].mean()):.1%}")

# the closed form for the area, checked against quadrature
area_quad = 0.5 * np.mean(D["r"]**2, axis=1) * 2 * np.pi
print(f"area: closed form vs quadrature, max difference = "
      f"{np.abs(area_quad - D['area']).max():.2e}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(11.2, 4.0), subplot_kw={"projection": "polar"})
for j, ax in enumerate(axes.ravel()):
    conv = D["convex"][j] > 0.5
    ax.plot(np.append(TH, TH[0]), np.append(D["r"][j], D["r"][j][0]),
            color=GEO_TEAL if conv else GEO_RUST, lw=1.6)
    ax.set_yticklabels([]); ax.set_xticklabels([]); ax.grid(alpha=0.2)
    ax.set_title(f"A={D['area'][j]:.2f}\nL={D['length'][j]:.2f}", fontsize=6.5, pad=2)
    ax.set_ylim(0, 1.8)
fig.suptitle("random plane curves — teal: convex,   rust: non-convex", y=1.02)
plt.tight_layout(); plt.show()

### Symmetry

Rotating a curve changes none of the three labels. The **coefficients**, however,
rotate: $r(\theta)\mapsto r(\theta-\theta_0)$ acts on the $k$-th mode by a rotation
through $k\theta_0$, so each mode is a $2$-dimensional real representation of
$SO(2)$. The invariant of a rotation is the radius, so the **power spectrum**
$p_k = a_k^2 + b_k^2$ is a complete set of invariants — and a model built on $p$ is
$SO(2)$-invariant *by construction*, with no augmentation and no learning.

In [ ]:
def rotate_coeffs(a, b, theta0):
    """Act on Fourier coefficients by rotating the curve through theta0."""
    c, s = np.cos(KS * theta0), np.sin(KS * theta0)
    return a * c - b * s, a * s + b * c


def power_spectrum(a, b):
    return a**2 + b**2


a_rot, b_rot = rotate_coeffs(D["a"], D["b"], 0.7)
print("under a rotation by 0.7 radians:")
print(f"   raw coefficients change by  {np.abs(a_rot - D['a']).max():.3f}")
print(f"   power spectrum changes by   {np.abs(power_spectrum(a_rot, b_rot) - power_spectrum(D['a'], D['b'])).max():.2e}")

# and the labels are genuinely invariant
D_rot = dict(D)
D_rot["a"], D_rot["b"] = a_rot, b_rot
r_rot = 1 + a_rot @ _COS + b_rot @ _SIN
print(f"   area recomputed from rotated curve differs by "
      f"{np.abs(0.5*np.mean(r_rot**2,axis=1)*2*np.pi - D['area']).max():.2e}")

This is Lecture 3's slide 9 in its strongest form. We now compare two feature maps on
identical data:

$$\phi_{\text{raw}}(C) = (a_1,\dots,a_K,b_1,\dots,b_K) \in \mathbb{R}^{2K},
\qquad
\phi_{\text{inv}}(C) = (p_1,\dots,p_K) \in \mathbb{R}^{K} .$$

The invariant map has *half* as many features. It will win by an enormous margin,
because it has been told something true about the problem.

---
## 6. Regression: area exactly, length identifiably

The area is $\pi a_0^2 + \frac{\pi}{2}\sum_k p_k$: a *linear* function of the
invariant features. So least squares on $\phi_{\text{inv}}$ should not merely fit
well — it should be exact, with recognisable coefficients. On the raw coefficients it
is hopeless: the area is even in $(a,b)$ while a linear model is odd about the
origin.

In [ ]:
def fit_ls(F, y):
    """Least squares with an intercept; returns (intercept, weights)."""
    F1 = np.hstack([np.ones((len(F), 1)), F])
    w, *_ = np.linalg.lstsq(F1, y, rcond=None)
    return w[0], w[1:]


def r2_score(F, y, w0, w):
    return 1 - np.mean((w0 + F @ w - y)**2) / np.var(y)


P = power_spectrum(D["a"], D["b"])            # invariant features
RAW = np.hstack([D["a"], D["b"]])             # non-invariant features

w0_inv, w_inv = fit_ls(P, D["area"])
w0_raw, w_raw = fit_ls(RAW, D["area"])

print("AREA")
print(f"  invariant features   R^2 = {r2_score(P, D['area'], w0_inv, w_inv):.12f}")
print(f"  raw coefficients     R^2 = {r2_score(RAW, D['area'], w0_raw, w_raw):.6f}")
print()
print(f"  learned intercept  {w0_inv:.10f}      pi = {np.pi:.10f}")
print(f"  learned weights    {np.array2string(w_inv, precision=8)}")
print(f"  pi/2 =             {np.pi/2:.8f}")
print(f"  max deviation from pi/2: {np.abs(w_inv - np.pi/2).max():.2e}")

$R^2 = 1$ to twelve decimals; the intercept is $\pi$ to ten; every one of the eight
weights is $\pi/2$ to thirteen. The model has not approximated the area — it has
**rediscovered the formula**, and its parameters are readable mathematics. That is
the claim in Lecture 3's closing block, and the reason to baseline every problem with
a linear model on thoughtful features. Meanwhile the raw-coefficient model, given
*more* features and the same data, achieves essentially nothing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.1))
axes[0].scatter(D["area"], w0_inv + P @ w_inv, s=3, alpha=0.35, color=GEO_TEAL, lw=0)
axes[0].plot([D["area"].min(), D["area"].max()], [D["area"].min(), D["area"].max()],
             color=GEO_RUST, lw=1.2)
axes[0].set_xlabel("true area"); axes[0].set_ylabel("predicted")
axes[0].set_title("invariant features: exact")

axes[1].plot(KS, w_inv, "o", ms=7, color=GEO_DARK, label="learned")
axes[1].axhline(np.pi / 2, color=GEO_RUST, lw=1.5, ls="--", label=r"$\pi/2$")
axes[1].set_xlabel("mode $k$"); axes[1].set_ylabel("coefficient")
axes[1].set_ylim(0, 2.4); axes[1].set_title("the learned weights"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Length: the same features, a much weaker conclusion

There is no closed form for $L$, but expanding to second order about the circle gives

$$L \;\approx\; 2\pi a_0 \;+\; \frac{\pi}{2a_0}\sum_k k^2 p_k ,$$

so with $a_0=1$ we predict an intercept of $2\pi$ and weights $\frac{\pi}{2}k^2$.
Those weights are worth reading: $\sum_k k^2 p_k$ is the **Dirichlet energy of the
boundary perturbation on $S^1$**, the $k^2$ being eigenvalues of $-d^2/d\theta^2$.
Part I's spectral weighting has reappeared as the answer to a regression about arc
length.

In [ ]:
w0_L, w_L = fit_ls(P, D["length"])
pred_w = (np.pi / 2) * KS**2

print("LENGTH")
print(f"  R^2 = {r2_score(P, D['length'], w0_L, w_L):.6f}")
print(f"  intercept {w0_L:.5f}   (2*pi = {2*np.pi:.5f})")
print(f"  raw-coefficient model R^2 = {r2_score(RAW, D['length'], *fit_ls(RAW, D['length'])):.6f}")
print()
print(f"  {'k':>3} {'learned':>10} {'(pi/2)k^2':>11} {'ratio':>8}   {'std of p_k':>11}")
for i, k in enumerate(KS):
    print(f"  {k:3d} {w_L[i]:10.3f} {pred_w[i]:11.3f} {w_L[i]/pred_w[i]:8.3f}   {P[:, i].std():11.2e}")

def design_kappa(F):
    F1 = np.hstack([np.ones((len(F), 1)), F])
    sv = np.linalg.svd(F1, compute_uv=False)
    return (sv[0] / sv[-1])**2


Z = (P - P.mean(axis=0)) / P.std(axis=0)     # standardised features
print(f"kappa, raw power spectrum   : {design_kappa(P):.3e}")
print(f"kappa, standardised         : {design_kappa(Z):.3e}")

# how uncertain is each weight? bootstrap.
B, n_sub = 60, 3000
boots = np.array([fit_ls(P[idx], D["length"][idx])[1]
                  for idx in (rng.integers(0, len(P), n_sub) for _ in range(B))])
lo, hi = np.percentile(boots, [5, 95], axis=0)

fig, ax = plt.subplots(figsize=(5.8, 3.4))
ax.plot(KS, pred_w, "-", color=GEO_RUST, lw=1.8, label=r"theory: $\frac{\pi}{2}k^2$")
ax.errorbar(KS, w_L, yerr=[w_L - lo, hi - w_L], fmt="o", color=GEO_DARK,
            capsize=3, ms=5, label="learned (90% bootstrap)")
ax.set_yscale("symlog", linthresh=1)
ax.set_xlabel("mode $k$"); ax.set_ylabel("coefficient")
ax.set_title("weights are identifiable only where the feature varies")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

The error bars tell the honest story: low-$k$ weights are pinned down and agree with
$\frac{\pi}{2}k^2$; high-$k$ confidence intervals are wide enough to swallow the
prediction. The model is not wrong about them — it is *uncertain*, and quoting those
numbers as discoveries would be a misreading.

Standardising collapses $\kappa$ from $\sim 10^5$ to about $1$ — Lecture 3's
"normalising features is literally rounding the ellipses". Note what that does and
does not fix: it makes the problem well-conditioned to *solve*, so gradient descent
converges fast. It does not manufacture information the data does not carry.

> **Exercise 4 — how much does invariance buy?**
> (a) A linear model on raw coefficients cannot represent the area; a *quadratic* one
> can, since $a_k^2, b_k^2$ are among its features. Fit it. How many features does it
> use, how many does it need, and what happens to the rest?
>
> (b) Train the quadratic raw model on curves in a fixed orientation and test on
> randomly rotated ones; then repeat with rotation augmentation. Compare both against
> the invariant model, which needs neither.
>
> (c) Relate what you find to Tutorial 2 Exercise 5. What is the analogue here of
> splitting by orbit rather than by sample?

---
## 7. Classification, ridge and lasso

Convexity of a star-shaped curve is a genuinely global property: no single
coefficient decides it. Following Lecture 3, model
$\mathbb{P}(\text{convex}\mid C) = \sigma(\theta^\top\phi(C))$ and minimise the
cross-entropy. It is convex, so plain gradient descent finds the global optimum; we
write the loop out because it is six lines and Lecture 3 derived every one.

In [ ]:
def sigmoid(t):
    return 0.5 * (1 + np.tanh(0.5 * t))       # numerically stable form


def fit_logistic(F, y, lam=0.0, iters=4000, eta=0.5):
    """Logistic regression by gradient descent on the cross-entropy."""
    F1 = np.hstack([np.ones((len(F), 1)), F])
    th = np.zeros(F1.shape[1])
    hist = []
    for _ in range(iters):
        p = sigmoid(F1 @ th)
        grad = F1.T @ (p - y) / len(y) + lam * np.r_[0.0, th[1:]]
        th -= eta * grad
        hist.append(-np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))
    return th, np.array(hist)


def predict_logistic(F, th):
    return sigmoid(np.hstack([np.ones((len(F), 1)), F]) @ th)


n_tr = 4000
tr, te = slice(0, n_tr), slice(n_tr, None)
Zp = (P - P[tr].mean(0)) / P[tr].std(0)        # standardise: kappa matters for GD
y_cv = D["convex"]

th_inv, hist_inv = fit_logistic(Zp[tr], y_cv[tr], eta=1.0)
Zr = (RAW - RAW[tr].mean(0)) / RAW[tr].std(0)
th_raw, hist_raw = fit_logistic(Zr[tr], y_cv[tr], eta=1.0)

acc = lambda F, th, y: np.mean((predict_logistic(F, th) > 0.5) == (y > 0.5))
base = max(y_cv[te].mean(), 1 - y_cv[te].mean())
print(f"majority-class baseline      {base:.1%}")
print(f"invariant features, test acc {acc(Zp[te], th_inv, y_cv[te]):.1%}")
print(f"raw coefficients,  test acc  {acc(Zr[te], th_raw, y_cv[te]):.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.2))

axes[0].plot(hist_inv, color=GEO_DARK, label="invariant")
axes[0].plot(hist_raw, color=GEO_RUST, label="raw")
axes[0].set_xlabel("gradient step"); axes[0].set_ylabel("cross-entropy")
axes[0].set_title("convex loss, monotone descent"); axes[0].legend(fontsize=8)

# calibration: predicted probability vs observed frequency
p_te = predict_logistic(Zp[te], th_inv)
bins = np.linspace(0, 1, 11)
idx = np.digitize(p_te, bins) - 1
obs = np.array([y_cv[te][idx == i].mean() if (idx == i).sum() > 20 else np.nan
                for i in range(10)])
axes[1].plot([0, 1], [0, 1], "--", color="0.5", lw=1)
axes[1].plot(0.5 * (bins[:-1] + bins[1:]), obs, "o-", color=GEO_TEAL)
axes[1].set_xlabel("predicted $\\mathbb{P}(\\mathrm{convex})$")
axes[1].set_ylabel("observed frequency")
axes[1].set_title("logistic outputs are calibrated")

# the two most informative features, with the decision boundary
w = th_inv[1:]
i, j = np.argsort(-np.abs(w))[:2]
sub = slice(n_tr, n_tr + 1500)
axes[2].scatter(Zp[sub, i], Zp[sub, j], c=y_cv[sub], cmap="coolwarm", s=6, lw=0, alpha=0.75)
xx = np.linspace(Zp[sub, i].min(), Zp[sub, i].max(), 50)
axes[2].plot(xx, -(th_inv[0] + w[i] * xx) / w[j], color=GEO_DARK, lw=1.8)
axes[2].set_xlabel(f"$p_{{{KS[i]}}}$ (standardised)"); axes[2].set_ylabel(f"$p_{{{KS[j]}}}$")
axes[2].set_ylim(Zp[sub, j].min(), Zp[sub, j].max())
axes[2].set_title("decision boundary is a hyperplane")
plt.tight_layout(); plt.show()

The middle panel is Lecture 3's point about logistic regression: the outputs are
**calibrated probabilities**, not just labels. When the model says $0.7$, about
$70\%$ of such curves really are convex — worth far more than a bare accuracy figure
if the model is going to suggest conjectures.

### Feature selection as conjecture generation

Lecture 3's slide 8 contrasted the penalties geometrically: the round $\ell^2$ ball
is touched at a smooth point, so every coefficient shrinks but none vanishes; the
$\ell^1$ ball has corners on the axes, so contact tends to happen there and
coefficients vanish exactly.

We use that as a *discovery* tool. Take the length regression, hand the model a pile
of candidates — the ones we believe in, some plausible alternatives, and some pure
noise — and ask lasso which it keeps.

In [ ]:
def soft_threshold(v, t):
    return np.sign(v) * np.maximum(np.abs(v) - t, 0.0)


def fit_lasso(F, y, lam, iters=4000):
    """Lasso by ISTA: a gradient step, then the proximal operator of the l1 norm."""
    n, d = F.shape
    step = 1.0 / np.linalg.norm(F, 2)**2 * n
    w = np.zeros(d); b = y.mean()
    for _ in range(iters):
        resid = F @ w + b - y
        w = soft_threshold(w - step * (F.T @ resid) / n, step * lam)
        b -= step * resid.mean()
    return b, w


# ---- candidate features: the truth, some decoys, and some noise ---------------
cand, names = [], []
for i, k in enumerate(KS):
    cand.append(P[:, i]); names.append(f"p_{k}")
cand.append((KS**2 * P).sum(1)); names.append("sum k^2 p_k  (Dirichlet)")
cand.append((KS**4 * P).sum(1)); names.append("sum k^4 p_k")
cand.append(P.sum(1));           names.append("sum p_k  (area)")
cand.append(np.sqrt(P.sum(1)));  names.append("sqrt(sum p_k)")
for j in range(6):
    cand.append(rng.normal(size=len(P))); names.append(f"noise_{j}")
F_all = np.stack(cand, axis=1)
F_all = (F_all - F_all.mean(0)) / F_all.std(0)      # lasso needs comparable scales
print(f"{F_all.shape[1]} candidate features, of which 6 are pure noise")

lams = np.logspace(-3.2, -0.4, 40)
path = np.array([fit_lasso(F_all[tr], D["length"][tr], l)[1] for l in lams])
val_mse = []
for l, w in zip(lams, path):
    b, _ = fit_lasso(F_all[tr], D["length"][tr], l)
    val_mse.append(np.mean((F_all[te] @ w + b - D["length"][te])**2))
val_mse = np.array(val_mse)
best = int(np.argmin(val_mse))

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.6))
for j in range(F_all.shape[1]):
    keep = np.abs(path[:, j]).max() > 1e-3
    axes[0].semilogx(lams, path[:, j], lw=1.6 if keep else 0.7,
                     color=None if keep else "0.75",
                     label=names[j] if keep else None)
axes[0].axvline(lams[best], color="0.4", ls="--", lw=1.2)
axes[0].set_xlabel(r"$\lambda$"); axes[0].set_ylabel("coefficient")
axes[0].set_title("lasso path: coefficients hit zero exactly")
axes[0].legend(fontsize=6.5, ncol=1)

axes[1].semilogx(lams, val_mse, "o-", ms=3, color=GEO_DARK)
axes[1].axvline(lams[best], color="0.4", ls="--", lw=1.2)
axes[1].set_xlabel(r"$\lambda$"); axes[1].set_ylabel("validation MSE")
axes[1].set_title(f"selected $\\lambda$ = {lams[best]:.2e}")
plt.tight_layout(); plt.show()

b_s, w_s = fit_lasso(F_all[tr], D["length"][tr], lams[best])
order = np.argsort(-np.abs(w_s))
print(f"features kept at the selected lambda ({int((np.abs(w_s) > 1e-6).sum())} of {len(w_s)}):")
for j in order:
    if abs(w_s[j]) > 1e-6:
        print(f"   {w_s[j]:+8.4f}   {names[j]}")
print("\nnoise features retained:",
      int(sum(abs(w_s[j]) > 1e-6 for j, nm in enumerate(names) if nm.startswith("noise"))))

Lasso discards the noise features and keeps a small set dominated by the Dirichlet
combination $\sum_k k^2 p_k$ — which is, as §6 argued, the correct second-order
answer. Had we not known the expansion, the sparse solution would have *suggested*
it: a readable claim about which geometric quantity controls arc length, obtained by
fitting.

That is "interpretability as conjecture generation" in practice. The usual cautions
apply: a selected feature is a hypothesis, not a theorem, and correlated candidates
make the choice among them unstable.

> **Exercise 5 — better features, and honest regularisation.**
> (a) Convexity is governed by $r^2+2r'^2-rr''$, whose $r''$ term weights mode $k$ by
> $k^2$. Add the features $k^2p_k$ and $k^4p_k$ and measure the gain. Which weighting
> matters most, and would you have guessed it?
>
> (b) Repeat the path with ridge and overlay the two. Confirm that ridge shrinks
> everything and zeroes nothing, and explain it from the geometry of the two balls.
>
> (c) Add a near-duplicate of $\sum_k k^2p_k$. Which does lasso choose, and how stable
> is the choice across bootstrap resamples? Show that the elastic net stabilises it.

---
## 8. The same fit in PyTorch

Everything above used closed forms and hand-written gradient descent, which is the
right way to learn it. From Lecture 4 the models will not have closed forms, so we
switch to the tool that does not need them.

Here is Lecture 3's canonical loop on the area regression — where we know the exact
answer, so we can check the machinery rather than trust it. The check is worth
making: it fails the first time, for a reason §6 already taught us.

In [ ]:
import torch


def train_linear(F, y, epochs=4000, lr=0.02, seed=0):
    """Lecture 3's canonical training loop, verbatim."""
    torch.manual_seed(seed)
    Xt = torch.tensor(F, dtype=torch.float64)
    yt = torch.tensor(y, dtype=torch.float64).unsqueeze(1)

    model = torch.nn.Linear(F.shape[1], 1, dtype=torch.float64)   # f(x) = Wx + b
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = torch.nn.MSELoss()

    for epoch in range(epochs):
        opt.zero_grad()                 # reset stored gradients
        loss = lossf(model(Xt), yt)     # forward pass
        loss.backward()                 # autodiff: compute grad L
        opt.step()                      # theta <- theta - eta * grad

    return (model.weight.detach().numpy().ravel(),
            float(model.bias.detach().numpy().item()),
            float(loss.detach()))


w_t, b_t, mse_t = train_linear(P, D["area"], epochs=20000)
print(f"training MSE {mse_t:.2e}   intercept {b_t:.6f}  (pi = {np.pi:.6f})")
print(f"  PyTorch     {np.array2string(w_t, precision=4)}")
print(f"  closed form {np.array2string(w_inv, precision=4)}")
print(f"  max |PyTorch weight - pi/2| = {np.abs(w_t - np.pi / 2).max():.3f}")

The training loss is tiny — and the answer is wrong. The first weight is $\pi/2$; the
rest are not, and some are wrong by more than half of $\pi/2$ itself. Training longer
does not rescue it.

This is §6's identifiability problem seen from the optimisation side. The feature
$p_8$ has standard deviation about $4\times10^{-6}$, so changing $w_8$ by a whole
unit moves the predictions by $4\times10^{-6}$ and the loss by around $10^{-11}$.
That direction of parameter space is *flat*: gradient descent has no signal to follow
and stops wherever it happens to be. The closed form escaped this only because the
system is consistent and a direct solve works at machine precision.

The fix is to make the problem well conditioned. Regenerate the curves with a
**flatter spectrum**, holding the optimiser, learning rate, epochs and target fixed,
so the only thing that changes is the conditioning:

In [ ]:
D_flat = make_curves(6000, rng, sigma=0.12, s=0.25)     # nearly flat spectral decay
P_flat = power_spectrum(D_flat["a"], D_flat["b"])
print(f"valid curves (r > 0): {(D_flat['r'] > 0).all(axis=1).mean():.1%}")
print(f"feature-scale spread: {P_flat.std(0).max() / P_flat.std(0).min():.1f}x"
      f"    (it was {P.std(0).max() / P.std(0).min():.0f}x)")

w_f, b_f, mse_f = train_linear(P_flat, D_flat["area"], epochs=20000)
print(f"\ntraining MSE {mse_f:.2e}   intercept {b_f:.6f}  (pi = {np.pi:.6f})")
print(f"  PyTorch weights {np.array2string(w_f, precision=6)}")
print(f"  max |w - pi/2| = {np.abs(w_f - np.pi / 2).max():.2e}")

Now the loop recovers $\pi/2$ in every coordinate and $\pi$ as the intercept, to
machine precision — the same answer the normal equations give, without ever forming
them.

Two lessons that outlive linear models:

- **A small training loss is not evidence that the parameters are right.** Along flat
  directions the loss is indifferent and the fitted values are arbitrary. When the
  coefficients are the point — as whenever a model is meant to suggest mathematics —
  check identifiability before reading them.
- **Conditioning is a property of the data and the feature map**, not of the
  optimiser. Adam's per-coordinate rescaling is a change of metric on parameter space
  (Lecture 2 §10) and it helps, but it cannot manufacture information the features do
  not carry.

> **Exercise 6 — the condition number, in training curves.**
> (a) Swap Adam for `torch.optim.SGD` and find the largest stable learning rate.
> Compare epochs-to-target with raw and with standardised features, and relate the
> ratio to the $\kappa$ values of §6.
>
> (b) Add `weight_decay` and confirm it reproduces the ridge solution of §7 for the
> corresponding $\lambda$.
>
> (c) Rewrite §7's logistic regression in PyTorch with `BCEWithLogitsLoss` and check
> it matches your hand-written gradient descent. Why is that loss preferred to a
> sigmoid followed by a plain cross-entropy?

---
## What to take away

- **Least squares is orthogonal projection**, on a manifold as much as in
  $\mathbb{R}^n$. The design matrix is a discretised Gram matrix, and having only
  samples means having only an approximate inner product.
- **The Laplacian's eigenbasis is the good basis.** It diagonalises the projection,
  makes $\kappa\approx1$ where monomials give $10^4$, and is recoverable from a bare
  point cloud — eigenvalues $0,2,6,12,20$, multiplicities $1,3,5,7,9$.
- **Regularisation is geometry.** Weighted ridge in the harmonic basis *is* a penalty
  on $\int|\nabla f|^2$, and its solution is a spectral filter. Choosing a penalty is
  choosing which functions you consider plausible.
- **Approximation error is coefficient decay**, hence regularity. No optimiser repairs
  a poor ansatz.
- **The feature map is the model.** On invariant features the area regression is exact
  and returns $\pi$ and $\pi/2$; on raw coordinates, with more parameters, nothing.
- **Read the error bars, not the point estimates.** Weights are identifiable only
  where their features vary.

### Next

**Lecture 4** replaces the fixed feature map with a learned one — neural networks —
giving up convexity, closed forms and guarantees for expressive power. **Tutorial 4**
compares the two on the same geometric data, so you can say what the trade bought.

### Further reading

- Atkinson & Han, *Spherical Harmonics and Approximation on the Unit Sphere* (Springer, 2012) — Part I in full rigour.
- Belkin & Niyogi, "Laplacian eigenmaps…", *Neural Computation* **15** (2003) — the construction in §3.
- Hastie, Tibshirani & Friedman, *The Elements of Statistical Learning*, ch. 3 — ridge, lasso, and the ball picture. Free online.
- Trefethen, *Approximation Theory and Approximation Practice*, ch. 7–8 — why smoothness governs decay.